#### Simple Gen AI APP Using Langchain

In [43]:
import os
import tqdm as notebook_tqdm
from dotenv import load_dotenv
load_dotenv()

## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]=os.getenv("LANGCHAIN_TRACING_V2")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [44]:
## Data Ingestion - Langchain websote doc link to scrape data from
from langchain_community.document_loaders import WebBaseLoader

In [45]:
loader = WebBaseLoader("https://www.langchain.com/blog/tuning-deep-agents-different-models")
loader

In [46]:
docs=loader.load()
print(f"Number of documents: {len(docs)}")
docs

Number of documents: 1


[Document(metadata={'source': 'https://www.langchain.com/blog/tuning-deep-agents-different-models', 'title': 'Tuning Deep Agents to Work Well with Different Models', 'description': 'Deep Agents was previously designed in a generic way to work well across model families. Today we’re adding model-specific profiles to adjust prompts, tools, and middleware. We ship profiles for OpenAI, Anthropic, and Google models out of the box, which we see leads to a 10–20 point jump on a subset of tau2-bench over the default harness.', 'language': 'en'}, page_content='Tuning Deep Agents to Work Well with Different Models\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceDeploymentShip and scale agents in productionFleetAgents for the whole companyOpen Source FrameworksdeepagentsBuild long-running agents for complex taskslangchainQuick

In [47]:
### Load Data => Docs => Divide into chunks documents => text => vectors => Vector Embedding => VectorDB => Retrieval => LLM response
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

In [48]:
documents

[Document(metadata={'source': 'https://www.langchain.com/blog/tuning-deep-agents-different-models', 'title': 'Tuning Deep Agents to Work Well with Different Models', 'description': 'Deep Agents was previously designed in a generic way to work well across model families. Today we’re adding model-specific profiles to adjust prompts, tools, and middleware. We ship profiles for OpenAI, Anthropic, and Google models out of the box, which we see leads to a 10–20 point jump on a subset of tau2-bench over the default harness.', 'language': 'en'}, page_content='Tuning Deep Agents to Work Well with Different Models\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nProducts\n\nLangSmith Platform\n\nObservabilitySee exactly what your agents are doingEvaluationScore and improve agent performanceDeploymentShip and scale agents in productionFleetAgents for the whole companyOpen Source FrameworksdeepagentsBuild long-running agents for complex taskslangchainQuick

In [49]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="gemma:2b")

In [50]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embeddings)
vectorstoredb

In [51]:
# Query from vectorstoredb
query = "What are the different models mentioned in the article?"
result=vectorstoredb.similarity_search(query)
result

[Document(id='bb729320-5c8b-4075-a0cb-ce606a619091', metadata={'source': 'https://www.langchain.com/blog/tuning-deep-agents-different-models', 'title': 'Tuning Deep Agents to Work Well with Different Models', 'description': 'Deep Agents was previously designed in a generic way to work well across model families. Today we’re adding model-specific profiles to adjust prompts, tools, and middleware. We ship profiles for OpenAI, Anthropic, and Google models out of the box, which we see leads to a 10–20 point jump on a subset of tau2-bench over the default harness.', 'language': 'en'}, page_content='Model\nBase Deep Agents Harness\nWith Custom Profile\n\n\n\n\nGPT 5.3 Codex\n33%\n53%\n\n\nClaude Opus 4.7\n43%\n53%'),
 Document(id='00146375-5990-43ef-a3dc-d12a6eea2352', metadata={'source': 'https://www.langchain.com/blog/tuning-deep-agents-different-models', 'title': 'Tuning Deep Agents to Work Well with Different Models', 'description': 'Deep Agents was previously designed in a generic way t

In [52]:
from langchain_ollama import ChatOllama 
llm=ChatOllama(model="gemma:2b")

In [55]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use the following retrieved documents to answer the question.

{context}

Question: {question}
Answer:
""")

doc_chain = (
    {
        "context": lambda x: x["context"],
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [58]:
doc_chain.invoke({
    "context": result,
    "question": query
})

AIMessage(content='OpenAI, Anthropic, and Google models are mentioned in the article.', additional_kwargs={}, response_metadata={'model': 'gemma:2b', 'created_at': '2026-05-04T00:26:11.541172Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2522319084, 'load_duration': 81671000, 'prompt_eval_count': 2874, 'prompt_eval_duration': 2222621917, 'eval_count': 16, 'eval_duration': 204277166, 'logprobs': None, 'model_name': 'gemma:2b', 'model_provider': 'ollama'}, id='lc_run--019df060-ccf8-7991-94ba-3f4dd20daa5d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2874, 'output_tokens': 16, 'total_tokens': 2890})

In [57]:
from langchain_core.documents import Document
doc_chain.invoke({
    "input": "What are the different models mentioned in the article?",
    "context"
: [Document(page_content="The article mentions several models, including the GPT-3 model and the Codex model.")]
})

AIMessage(content='The different models mentioned in the article are:\n\n* GPT-3 model\n* Codex model', additional_kwargs={}, response_metadata={'model': 'gemma:2b', 'created_at': '2026-05-04T00:24:14.058938Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1374751541, 'load_duration': 1065371416, 'prompt_eval_count': 117, 'prompt_eval_duration': 136012666, 'eval_count': 20, 'eval_duration': 155740584, 'logprobs': None, 'model_name': 'gemma:2b', 'model_provider': 'ollama'}, id='lc_run--019df05f-0689-7c60-a258-5a68518fa7f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 20, 'total_tokens': 137})

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [59]:
### Input--->Retriever--->vectorstoredb
vectorstoredb

In [ ]:
from langchain_core.runnables import RunnablePassthrough

retrieval_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | doc_chain
)

In [62]:
retrieval_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x162780180>, search_kwargs={}),
  question: RunnablePassthrough()
}
| {
    context: RunnableLambda(...),
    question: RunnablePassthrough()
  }
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Use the following retrieved documents to answer the question.\n\n{context}\n\nQuestion: {question}\nAnswer:\n'), additional_kwargs={})])
| ChatOllama(output_version=None, model='gemma:2b')

In [74]:
## Get the response form the LLM
response=retrieval_chain.invoke("What are the different models mentioned in the article?")
response

AIMessage(content='"OpenAI, Anthropic, and Google"\n</start_of_turn>', additional_kwargs={}, response_metadata={'model': 'gemma:2b', 'created_at': '2026-05-04T01:05:01.592226Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2433341458, 'load_duration': 60109750, 'prompt_eval_count': 2874, 'prompt_eval_duration': 2207300417, 'eval_count': 19, 'eval_duration': 154320084, 'logprobs': None, 'model_name': 'gemma:2b', 'model_provider': 'ollama'}, id='lc_run--019df084-5b15-71a0-861a-bc4b411d64ac-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2874, 'output_tokens': 19, 'total_tokens': 2893})

In [72]:
print(response.content)

The different models mentioned in the article are OpenAI, Anthropic, and Google.
